# Reporte Stage 2 — Frozen v1 vs Frozen v2 (IL-CLDM)

Comparación de dos variantes del LDM con encoder MRI **congelado**:

| Modo | Experimento | Run inferencia | Early stop |
|------|-------------|----------------|------------|
| frozen_v1 | `exp_nm_s2_mri_frozen` | `nm_stage2_v1` | PSNR + SSIM×10 (imagen) |
| frozen_v2 | `exp_nm_s2_mri_frozen_v2` | `nm_stage2_v2` | menor `latent_MSE` (val) |

**Alcance:** split **test** (36 sujetos).

Secciones:
1. Convergencia en entrenamiento (`loss_curve`, `validation`, `validation_latent`, checkpoints)
2. Reconstrucción en test (inferencia) + métricas ROI
3. Visualizaciones comparables (mismos sujetos)
4. Clasificación en test (`classification_metrics.csv`)
5. Síntesis
6. Diagnóstico sampling (`t` fijo vs DDIM 100) — v1 y v2


## 0. Configuración

In [ ]:
import os
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns


def find_repo_root() -> Path:
    here = Path.cwd().resolve()
    for candidate in (here, here.parent):
        if (candidate / "result").is_dir() and (candidate / "code" / "stage2_report_utils.py").is_file():
            return candidate
    raise RuntimeError(
        "No se encontró la raíz del repo. Ejecuta el notebook desde IL-CLDM/ o IL-CLDM/code/."
    )


ROOT = find_repo_root()
os.chdir(ROOT)

root_str = str(ROOT)
if root_str in sys.path:
    sys.path.remove(root_str)
sys.path.insert(0, root_str)

if "code" in sys.modules and not hasattr(sys.modules["code"], "__path__"):
    del sys.modules["code"]

from code.stage2_report_utils import (
    FROZEN_COMPARISON,
    LABEL_NAMES,
    available_comparison,
    comparison_entry,
    load_classification_metrics_for,
    load_inference_reconstruction_for,
    load_run_manifest,
    load_synthetic_volume_for,
    load_test_ids,
    load_training_frames_for,
    manifest_summary_table,
    pick_stratified_subjects,
    repo_root,
    training_checkpoint_summary,
    validate_frozen_comparison,
)
from viz_pairs import display_range, load_cropped

# Solo experimentos del registro con resultados de entrenamiento en disco
# (las ablaciones aun no ejecutadas se omiten automaticamente).
COMPARISON = available_comparison()
MODES = list(COMPARISON.keys())
MODE_LABELS = {k: COMPARISON[k]["label"] for k in MODES}
_PALETTE = ["#55A868", "#8172B2", "#C44E52", "#4C72B0", "#DD8452", "#937860"]
MODE_COLORS = {m: _PALETTE[i % len(_PALETTE)] for i, m in enumerate(MODES)}
MODE_ORDER = [MODE_LABELS[m] for m in MODES]
MODE_PALETTE = [MODE_COLORS[m] for m in MODES]

DATA_ROOT = repo_root() / "data"
TEST_IDS = load_test_ids()

sns.set_theme(style="whitegrid", context="notebook")
%matplotlib inline

print(f"Repo root: {repo_root()}")
print(f"Test subjects: {len(TEST_IDS)}")
for mode in MODES:
    cfg = COMPARISON[mode]
    print(f"  {MODE_LABELS[mode]}: {cfg['exp']} / {cfg['run_id']}")
for w in validate_frozen_comparison():
    print(f"WARNING: {w}")


## 0b. Resumen de configuración (registro + manifests)

Fuente única de verdad: `experiments/registry.json`. Cada corrida de entrenamiento escribe `result/<exp>/run_manifest.json` (flags activos, seed, commit, GPU), de modo que cada experimento queda autodescrito.

**Criterios de éxito / fallo por ablación** (todas parten de `frozen_v1`, early stop por imagen):

| Ablación | Cambio único | ÉXITO | FALLO |
|----------|--------------|-------|-------|
| `abl_latloss01` | `LATENT_LOSS_LAMBDA=0.1` | PSNR test ≥ frozen_v1 (margen 0.1 dB) **y** `latent_MSE` val mejora | PSNR cae >0.5 dB **o** curvas divergen |
| `abl_hight05` | `HIGH_T_SAMPLE_PROB=0.5` | mejora `latent_MSE` en `fixed_t` altos (t≥700) **y** PSNR test ≥ frozen_v1 | sin mejora en diagnóstico **o** PSNR cae >0.5 dB |
| `abl_auxfreq5` | `AUX_IMAGE_EVERY_N=5` | SSIM test ≥ frozen_v1 (margen 0.01) **y** PSNR test ≥ frozen_v1 sin inestabilidad en `aux_image_loss` | PSNR/SSIM caen >0.5 dB / 0.01 **o** curva aux con picos erráticos |


In [ ]:
config_table = manifest_summary_table()
print(f"Experimentos en el registro: {len(config_table)}")
print(f"Con resultados de entrenamiento (manifest): {int(config_table['has_manifest'].sum())}")
config_table

## 1. Convergencia durante entrenamiento

Fuentes en `result/<exp>/`:
- `loss_curve.csv`: MSE difusión + aux image (+ `latent_loss` en v2)
- `validation.csv` / `validation_latent.csv`: métricas cada ~5 épocas
- `test.csv` / `test_latent.csv`: evaluación en test **solo cuando mejora** el criterio de checkpoint
  - **v1:** mayor PSNR + SSIM×10 en val
  - **v2:** menor `latent_MSE` en val


In [ ]:
training = {mode: load_training_frames_for(COMPARISON[mode]["exp"]) for mode in MODES}

n_aux = sum(1 for m in MODES if "aux_image_loss" in training[m].get("loss_curve", pd.DataFrame()).columns)
n_latent = sum(1 for m in MODES if "latent_loss" in training[m].get("loss_curve", pd.DataFrame()).columns)
n_plots = 1 + int(n_aux > 0) + int(n_latent > 0)
fig, axes = plt.subplots(1, n_plots, figsize=(5 * n_plots, 4))
if n_plots == 1:
    axes = [axes]

pi = 0
for mode in MODES:
    loss = training[mode]["loss_curve"]
    axes[pi].plot(loss["Epoch"], loss["MSE_loss"], label=MODE_LABELS[mode], color=MODE_COLORS[mode])
axes[pi].set(xlabel="Época", ylabel="MSE loss", title="Pérdida de difusión (MSE)")
axes[pi].legend()
pi += 1

if n_aux > 0:
    for mode in MODES:
        loss = training[mode]["loss_curve"]
        if "aux_image_loss" in loss.columns:
            axes[pi].plot(loss["Epoch"], loss["aux_image_loss"], label=MODE_LABELS[mode], color=MODE_COLORS[mode])
    axes[pi].set(xlabel="Época", ylabel="Aux image loss", title="Pérdida auxiliar de imagen")
    axes[pi].legend()
    pi += 1

if n_latent > 0:
    for mode in MODES:
        loss = training[mode]["loss_curve"]
        if "latent_loss" in loss.columns:
            axes[pi].plot(loss["Epoch"], loss["latent_loss"], label=MODE_LABELS[mode], color=MODE_COLORS[mode])
    axes[pi].set(xlabel="Época", ylabel="Latent loss", title="Pérdida directa en latente (v2)")
    axes[pi].legend()

plt.tight_layout()
plt.show()


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
for mode in MODES:
    val = training[mode]["validation"]
    axes[0].plot(val["Epoch"], val["PSNR"], marker="o", ms=3, label=MODE_LABELS[mode], color=MODE_COLORS[mode])
    axes[1].plot(val["Epoch"], val["SSIM"], marker="o", ms=3, label=MODE_LABELS[mode], color=MODE_COLORS[mode])
axes[0].set(xlabel="Época", ylabel="PSNR (dB)", title="Validación — PSNR (imagen)")
axes[1].set(xlabel="Época", ylabel="SSIM", title="Validación — SSIM (imagen)")
for ax in axes:
    ax.legend()
plt.tight_layout()
plt.show()

if any("validation_latent" in training[m] for m in MODES):
    fig, axes = plt.subplots(1, 2, figsize=(14, 4))
    for mode in MODES:
        if "validation_latent" not in training[mode]:
            continue
        vl = training[mode]["validation_latent"]
        axes[0].plot(vl["Epoch"], vl["latent_MSE"], marker="o", ms=3, label=MODE_LABELS[mode], color=MODE_COLORS[mode])
        axes[1].plot(vl["Epoch"], vl["latent_MAE"], marker="o", ms=3, label=MODE_LABELS[mode], color=MODE_COLORS[mode])
    axes[0].set(xlabel="Época", ylabel="latent_MSE", title="Validación — MSE latente (v2 criterion)")
    axes[1].set(xlabel="Época", ylabel="latent_MAE", title="Validación — MAE latente")
    for ax in axes:
        ax.legend()
    plt.tight_layout()
    plt.show()


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
for mode in MODES:
    val = training[mode]["validation"]
    test_ckpt = training[mode]["test"]
    axes[0].plot(val["Epoch"], val["PSNR"], alpha=0.5, color=MODE_COLORS[mode])
    axes[0].scatter(
        test_ckpt["Epoch"], test_ckpt["PSNR"],
        s=60, marker="*", color=MODE_COLORS[mode],
        label=f"{MODE_LABELS[mode]} (test @ mejora)", zorder=3,
    )
    if "validation_latent" in training[mode]:
        vl = training[mode]["validation_latent"]
        axes[1].plot(vl["Epoch"], vl["latent_MSE"], alpha=0.5, color=MODE_COLORS[mode])
        if "test_latent" in training[mode]:
            tl = training[mode]["test_latent"]
            axes[1].scatter(
                tl["Epoch"], tl["latent_MSE"],
                s=60, marker="*", color=MODE_COLORS[mode], zorder=3,
            )
axes[0].set(xlabel="Época", ylabel="PSNR (dB)", title="Checkpoints sobre curva val (imagen)")
axes[1].set(xlabel="Época", ylabel="latent_MSE", title="Checkpoints sobre curva val (latente)")
axes[0].legend()
plt.tight_layout()
plt.show()


In [ ]:
ckpt_rows = []
for mode in MODES:
    cfg = COMPARISON[mode]
    criterion = cfg.get("early_stop", "image")
    summary = training_checkpoint_summary(
        training[mode]["validation"],
        training[mode]["test"],
        criterion=criterion,
        validation_latent=training[mode].get("validation_latent"),
        test_latent=training[mode].get("test_latent"),
    )
    summary["mode"] = MODE_LABELS[mode]
    ckpt_rows.append(summary)

ckpt_df = pd.DataFrame(ckpt_rows)
cols = [
    "mode", "early_stop_criterion", "last_epoch", "best_epoch",
    "val_psnr", "val_ssim", "val_latent_mse", "val_latent_mae",
    "test_psnr_at_best", "test_ssim_at_best",
    "test_latent_mse_at_best", "test_latent_mae_at_best",
    "n_val_points", "n_test_checkpoints",
]
ckpt_df = ckpt_df[[c for c in cols if c in ckpt_df.columns]]
ckpt_df


## 2. Reconstrucción en test (inferencia)

Fuente: `result/<exp>/inference/<run_id>/reconstruction_metrics.csv` (`split == test`).
v2 incluye columnas opcionales `PSNR_roi` / `SSIM_roi` (ganglios basales).


In [ ]:
recon_by_mode = {
    mode: load_inference_reconstruction_for(
        COMPARISON[mode]["exp"], COMPARISON[mode]["run_id"], mode
    )
    for mode in MODES
}
recon_all = pd.concat(recon_by_mode.values(), ignore_index=True)
recon_all["mode_label"] = recon_all["mode"].map(MODE_LABELS)

metric_cols = ["PSNR", "SSIM"]
if "PSNR_roi" in recon_all.columns:
    metric_cols.extend(["PSNR_roi", "SSIM_roi"])

summary_recon = (
    recon_all.groupby("mode_label")[metric_cols]
    .agg(["mean", "std", "min", "max"])
    .round(3)
)
summary_recon


In [ ]:
n_metrics = 2 + int("PSNR_roi" in recon_all.columns) * 2
fig, axes = plt.subplots(1, n_metrics, figsize=(5 * n_metrics, 4))
if n_metrics == 1:
    axes = [axes]
plot_metrics = [("PSNR", "PSNR en test"), ("SSIM", "SSIM en test")]
if "PSNR_roi" in recon_all.columns:
    plot_metrics.extend([("PSNR_roi", "PSNR ROI"), ("SSIM_roi", "SSIM ROI")])
for ax, (col, title) in zip(axes, plot_metrics):
    sns.boxplot(data=recon_all, x="mode_label", y=col, order=MODE_ORDER, ax=ax, palette=MODE_PALETTE)
    ax.set_title(title)
    ax.set_xlabel("")
plt.tight_layout()
plt.show()


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for label_name in ["control", "parkinson"]:
    subset = recon_all[recon_all["label_name"] == label_name]
    ax = axes[0] if label_name == "control" else axes[1]
    sns.boxplot(data=subset, x="mode_label", y="PSNR", order=MODE_ORDER, ax=ax, palette=MODE_PALETTE)
    ax.set_title(f"PSNR por experimento — {label_name}")
    ax.set_xlabel("")
plt.tight_layout()
plt.show()

In [ ]:
pivot_psnr = recon_all.pivot(index="ID", columns="mode", values="PSNR")
pivot_psnr["mean"] = pivot_psnr.mean(axis=1)
pivot_psnr = pivot_psnr.sort_values("mean", ascending=False)
pivot_psnr.columns = [MODE_LABELS.get(c, c) for c in pivot_psnr.columns]
pivot_psnr.head(10)

In [ ]:
v1 = recon_by_mode["frozen_v1"].set_index("ID")
v2 = recon_by_mode["frozen_v2"].set_index("ID")
common = v1.index.intersection(v2.index)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, col, ylab in zip(axes, ["PSNR", "SSIM"], ["PSNR v2", "SSIM v2"]):
    x = v1.loc[common, col]
    y = v2.loc[common, col]
    ax.scatter(x, y, alpha=0.75)
    lims = [min(x.min(), y.min()) - 0.5, max(x.max(), y.max()) + 0.5]
    ax.plot(lims, lims, "k--", alpha=0.4)
    ax.set(xlabel=f"{col} frozen v1", ylabel=ylab, title=f"{col} por sujeto: v1 vs v2")
    wins = (y > x).sum() if col != "SSIM" else (y > x).sum()
    ax.text(0.05, 0.95, f"v2 mejor en {wins}/{len(common)} sujetos", transform=ax.transAxes, va="top")
plt.tight_layout()
plt.show()


## 3. Visualizaciones comparables

6 sujetos test: 3 control + 3 parkinson (p10/p50/p90 PSNR medio entre v1 y v2).
Columnas: GT SPECT | Frozen v1 | Frozen v2 (corte axial central).


In [ ]:
selected = pick_stratified_subjects(recon_by_mode)
selected[["ID", "label_name", "tier", "psnr_mean"] + [c for c in selected.columns if c.startswith("psnr_")]]

In [ ]:
def plot_comparison_grid(selected_df: pd.DataFrame) -> None:
    col_titles = ["GT SPECT"] + [MODE_LABELS[m] for m in MODES]
    n_cols = len(col_titles)
    n_rows = len(selected_df)
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(4.5 * n_cols, 3.2 * n_rows))
    if n_rows == 1:
        axes = np.expand_dims(axes, axis=0)

    for row_idx, (_, row) in enumerate(selected_df.iterrows()):
        sid = str(row["ID"])
        gt = load_cropped(DATA_ROOT / "whole_Abeta", sid)
        z = gt.shape[2] // 2
        gt_lim = display_range(gt, symmetric=False, percentiles=(5.0, 99.5))

        volumes = [gt]
        for mode in MODES:
            cfg = COMPARISON[mode]
            volumes.append(load_synthetic_volume_for(cfg["exp"], cfg["run_id"], sid))

        for col_idx, (vol, title) in enumerate(zip(volumes, col_titles)):
            ax = axes[row_idx, col_idx]
            lim = gt_lim if col_idx == 0 else display_range(vol, symmetric=False, percentiles=(5.0, 99.5))
            ax.imshow(vol[:, :, z].T, cmap="hot", origin="lower", vmin=lim[0], vmax=lim[1])
            if row_idx == 0:
                ax.set_title(title, fontsize=10)
            ax.axis("off")
            if col_idx == 0:
                ax.set_ylabel(f"{sid}\n{row['label_name']}\n{row['tier']}", fontsize=9)

        metrics_txt = " | ".join(
            f"{MODE_LABELS[m][:6]} PSNR={recon_by_mode[m].loc[recon_by_mode[m]['ID']==sid, 'PSNR'].iloc[0]:.1f}"
            for m in MODES
        )
        fig.text(0.02, 1 - (row_idx + 0.92) / n_rows, metrics_txt, fontsize=8, va="top")

    fig.suptitle("Reconstrucciones SPECT — frozen v1 vs v2 (test)", y=1.01)
    plt.tight_layout()
    plt.show()


plot_comparison_grid(selected)


## 4. Clasificación en test

Fuente única: `classification/classification_metrics.csv` por experimento.
Los clasificadores entrenan con latentes **train** y evalúan en latentes **test** generados por inferencia.

In [ ]:
cls_frames = [
    load_classification_metrics_for(COMPARISON[m]["exp"], COMPARISON[m]["run_id"], m)
    for m in MODES
]
cls_all = pd.concat(cls_frames, ignore_index=True)
cls_all["mode_label"] = cls_all["mode"].map(MODE_LABELS)

cls_pivot = cls_all.pivot_table(
    index="model",
    columns="mode_label",
    values=["accuracy", "balanced_accuracy", "f1", "roc_auc"],
).round(3)
cls_pivot


In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
sns.barplot(
    data=cls_all,
    x="model",
    y="balanced_accuracy",
    hue="mode_label",
    hue_order=MODE_ORDER,
    palette=MODE_PALETTE,
    ax=ax,
)
ax.set_title("Balanced accuracy en test (clasificación de latentes)")
ax.set_xlabel("Modelo")
ax.set_ylabel("Balanced accuracy")
ax.tick_params(axis="x", rotation=30)
plt.tight_layout()
plt.show()

In [ ]:
best_per_mode = (
    cls_all.sort_values("balanced_accuracy", ascending=False)
    .groupby("mode_label", as_index=False)
    .first()
    [["mode_label", "model", "accuracy", "balanced_accuracy", "f1", "roc_auc"]]
    .round(3)
)
best_per_mode

## 5. Síntesis y exportación

In [ ]:
out_dir = repo_root() / "result" / "stage2_frozen_v1_v2_summary"
out_dir.mkdir(parents=True, exist_ok=True)

recon_summary = (
    recon_all.groupby(["mode", "mode_label", "label_name"])[metric_cols]
    .agg(["mean", "std", "count"])
    .round(4)
)
recon_summary.to_csv(out_dir / "reconstruction_test_summary.csv")
cls_all.to_csv(out_dir / "classification_comparison.csv", index=False)
ckpt_df.to_csv(out_dir / "training_checkpoint_summary.csv", index=False)
selected.to_csv(out_dir / "selected_visualization_subjects.csv", index=False)

print(f"Exportado en {out_dir}")
print("\n--- Resumen rápido ---")
for mode in MODES:
    r = recon_by_mode[mode]
    b = best_per_mode[best_per_mode["mode_label"] == MODE_LABELS[mode]].iloc[0]
    line = (
        f"{MODE_LABELS[mode]}: PSNR test={r['PSNR'].mean():.2f}±{r['PSNR'].std():.2f}, "
        f"mejor clf={b['model']} (bal.acc={b['balanced_accuracy']:.3f})"
    )
    if "PSNR_roi" in r.columns:
        line += f", PSNR_roi={r['PSNR_roi'].mean():.2f}"
    print(line)


## 6. Diagnóstico sampling: `t` fijo vs DDIM

Compara **frozen v1** y **frozen v2** (mismo protocolo por experimento):

| Familia | Descripción |
|---------|-------------|
| **A. t fijo** | `z_gt` corrompido → un paso `predict_x0` |
| **B. DDIM** | Ruido puro → `sampling_steps` iteraciones |

Configura `DIAG_MODES` (subconjunto de `MODES`). **Nota:** DDIM 200 fuera de alcance v2; se usa solo 100.


In [ ]:
import os
os.environ["CUDA_DEVICE_ORDER"] = "PCI_BUS_ID";
os.environ["CUDA_VISIBLE_DEVICES"] = "6"

In [ ]:
import json
from pathlib import Path

import torch
from torch.utils.data import DataLoader

import config
from dataset import TwoDataset, crop, nifti_to_numpy, resolve_nifti_path
from main import Diffusion, batch_metric_scores
from model import AAE, UNet
from utils import load_checkpoint, seed_torch
from inference import (
    checkpoint_paths,
    load_mri_encoder,
    resolve_mri_encoder_checkpoint,
    subject_id_from_name,
)

DIAG_MODES = ["frozen_v1", "frozen_v2"]  # o ["frozen_v1"] para prueba rápida
DIAG_MODE = "frozen"
DIAG_SEED = 42
MAX_SUBJECTS = 10
FIXED_T_VALUES = [50, 200, 500, 700, 999]
DDIM_STEPS_LIST = [100]
USE_LABEL = True

device = torch.device(config.device)
seed_torch(DIAG_SEED)


def load_diag_models(mode_key: str):
    cfg = COMPARISON[mode_key]
    exp = cfg["exp"]
    run_id = cfg["run_id"]
    run_cfg_path = repo_root() / "result" / exp / "inference" / run_id / "run_config.json"
    if run_cfg_path.exists():
        with open(run_cfg_path) as f:
            run_cfg = json.load(f)
        checkpoint_aae = repo_root() / run_cfg["checkpoints"]["AAE"]
        checkpoint_unet = repo_root() / run_cfg["checkpoints"]["Unet"]
        mri_ckpt = repo_root() / run_cfg["checkpoints"]["MRI_encoder"]
    else:
        checkpoint_aae, checkpoint_unet = checkpoint_paths(exp)
        checkpoint_aae = Path(checkpoint_aae)
        checkpoint_unet = Path(checkpoint_unet)
        mri_ckpt = resolve_mri_encoder_checkpoint(exp, DIAG_MODE, None)

    aae = AAE().to(device).eval()
    load_checkpoint(str(checkpoint_aae), aae, None, config.learning_rate)
    unet = UNet().to(device).eval()
    load_checkpoint(str(checkpoint_unet), unet, None, config.learning_rate)
    mri_encoder, mri_ckpt_resolved = load_mri_encoder(DIAG_MODE, mri_ckpt, device)
    diffusion = Diffusion()
    print(f"[{MODE_LABELS[mode_key]}] exp={exp} UNet={checkpoint_unet}")
    return aae, unet, mri_encoder, diffusion, exp

print(f"Variantes: {[MODE_LABELS[m] for m in DIAG_MODES]} | DDIM steps: {DDIM_STEPS_LIST}")



def _latent_metrics(z_hat: torch.Tensor, z_gt: torch.Tensor) -> dict:
    a = z_hat.detach().float().cpu().numpy().ravel()
    b = z_gt.detach().float().cpu().numpy().ravel()
    mse = float(np.mean((a - b) ** 2))
    mae = float(np.mean(np.abs(a - b)))
    corr = float(np.corrcoef(a, b)[0, 1]) if a.size > 1 else float("nan")
    return {"latent_mse": mse, "latent_mae": mae, "latent_corr": corr}


def _image_metrics(aae_model, spect_gt: torch.Tensor, z: torch.Tensor) -> dict:
    with torch.no_grad():
        pred = aae_model.decoder(z)
    psnr_v, ssim_v = batch_metric_scores(spect_gt.detach().cpu(), pred.detach().cpu())
    return {"PSNR": float(psnr_v), "SSIM": float(ssim_v)}


def _mri_condition(mri_enc, MRI: torch.Tensor) -> tuple[torch.Tensor, bool]:
    if mri_enc is not None:
        with torch.no_grad():
            return mri_enc(MRI), True
    return MRI, False


def _predict_noise(unet_model, x_t, cond, t, label, precomputed: bool):
    if USE_LABEL:
        return unet_model(x_t, cond, t, label, precomputed_condition=precomputed)
    return unet_model(x_t, cond, t, label=None, precomputed_condition=precomputed)


def _ddim_sample(unet_model, diff_model, cond, precomputed, label_batch, steps: int, seed: int) -> torch.Tensor:
    seed_torch(seed)

    def _model(x, y, t):
        return _predict_noise(unet_model, x, y, t, label_batch, precomputed)

    return diff_model.sample(_model, cond, sampling_steps=steps, eta=config.ddim_eta)


test_ds = TwoDataset(
    root_MRI=str(DATA_ROOT / "whole_MRI"),
    root_Abeta=str(DATA_ROOT / "whole_Abeta"),
    task=str(repo_root() / "data_info" / "test.txt"),
    stage="test",
)
if MAX_SUBJECTS is not None:
    test_ds.images = test_ds.images[:MAX_SUBJECTS]
    test_ds.length_dataset = len(test_ds.images)
    test_ds.len = len(test_ds.images)

loader = DataLoader(test_ds, batch_size=1, shuffle=False, num_workers=0)

from tqdm.auto import tqdm

diag_rows = []
for mode_key in DIAG_MODES:
    aae, unet, mri_encoder, diffusion, diag_exp = load_diag_models(mode_key)
    variant_label = MODE_LABELS[mode_key]

    for batch_idx, (MRI, Abeta, name, label) in enumerate(
        tqdm(loader, desc=f"sampling diagnosis [{variant_label}]")
    ):
        sid = subject_id_from_name(name)
        MRI_t = torch.as_tensor(np.expand_dims(MRI, 1), dtype=torch.float32, device=device)
        spect_gt = torch.as_tensor(np.expand_dims(Abeta, 1), dtype=torch.float32, device=device)
        label_batch = label.to(device)

        with torch.no_grad():
            z_gt = aae.encoder(spect_gt)
            cond, precomputed = _mri_condition(mri_encoder, MRI_t)

            m_oracle = {**_latent_metrics(z_gt, z_gt), **_image_metrics(aae, spect_gt, z_gt)}
            diag_rows.append({
                "ID": sid,
                "variant": mode_key,
                "variant_label": variant_label,
                "method": "oracle_z_gt",
                "t_or_steps": None,
                **m_oracle,
            })

            for t_val in FIXED_T_VALUES:
                t = torch.full((1,), int(t_val), dtype=torch.long, device=device)
                x_t, _noise = diffusion.noise_images(z_gt, t)
                eps_pred = _predict_noise(unet, x_t, cond, t, label_batch, precomputed)
                z_hat = diffusion.predict_x0(x_t, t, eps_pred)
                m = {**_latent_metrics(z_hat, z_gt), **_image_metrics(aae, spect_gt, z_hat)}
                diag_rows.append({
                    "ID": sid,
                    "variant": mode_key,
                    "variant_label": variant_label,
                    "method": "fixed_t",
                    "t_or_steps": t_val,
                    **m,
                })

            for steps in DDIM_STEPS_LIST:
                z_gen = _ddim_sample(
                    unet, diffusion, cond, precomputed, label_batch, steps,
                    seed=DIAG_SEED + batch_idx,
                )
                m = {**_latent_metrics(z_gen, z_gt), **_image_metrics(aae, spect_gt, z_gen)}
                diag_rows.append({
                    "ID": sid,
                    "variant": mode_key,
                    "variant_label": variant_label,
                    "method": "ddim",
                    "t_or_steps": steps,
                    **m,
                })

diag_df = pd.DataFrame(diag_rows)
diag_df["use_label"] = USE_LABEL
diag_df["experiment"] = diag_df["variant"].map(lambda k: COMPARISON[k]["exp"])
print(
    f"Filas: {len(diag_df)} | sujetos: {diag_df['ID'].nunique()} | "
    f"variantes: {diag_df['variant_label'].unique().tolist()}"
)
diag_df.head(8)


In [ ]:
# Esta celda se fusionó con la anterior (setup + diagnóstico en una sola).
# Ejecuta solo la celda de setup de la sección 6.
assert 'diag_df' in globals(), 'Ejecuta primero la celda fusionada de setup/diagnóstico'
diag_df.head(8)

In [ ]:
def _method_label(row):
    if row["method"] == "oracle_z_gt":
        return "oracle z_gt"
    if row["method"] == "fixed_t":
        return f"t={int(row['t_or_steps'])}"
    return f"DDIM-{int(row['t_or_steps'])}"


diag_df["method_label"] = diag_df.apply(_method_label, axis=1)

summary = (
    diag_df.groupby(["variant_label", "method", "t_or_steps", "method_label"], dropna=False)
    .agg(
        n=("ID", "count"),
        latent_mse_mean=("latent_mse", "mean"),
        latent_mse_std=("latent_mse", "std"),
        latent_corr_mean=("latent_corr", "mean"),
        PSNR_mean=("PSNR", "mean"),
        PSNR_std=("PSNR", "std"),
        SSIM_mean=("SSIM", "mean"),
        SSIM_std=("SSIM", "std"),
    )
    .reset_index()
    .sort_values(["variant_label", "method", "t_or_steps"])
    .round(4)
)
summary


In [ ]:
plot_order = (
    ["oracle z_gt"]
    + [f"t={t}" for t in FIXED_T_VALUES]
    + [f"DDIM-{s}" for s in DDIM_STEPS_LIST]
)

for variant_label in summary["variant_label"].unique():
    plot_df = summary[summary["variant_label"] == variant_label].copy()
    plot_df = plot_df[plot_df["method_label"].isin(plot_order)].copy()
    plot_df["method_label"] = pd.Categorical(plot_df["method_label"], categories=plot_order, ordered=True)
    plot_df = plot_df.sort_values("method_label")

    fig, axes = plt.subplots(1, 3, figsize=(16, 4))
    x = np.arange(len(plot_df))
    labels = plot_df["method_label"].astype(str)

    axes[0].bar(x, plot_df["latent_mse_mean"], yerr=plot_df["latent_mse_std"], capsize=3, color="steelblue")
    axes[0].set_title("MSE latente vs z_gt")
    axes[0].set_xticks(x)
    axes[0].set_xticklabels(labels, rotation=45, ha="right")

    axes[1].bar(x, plot_df["PSNR_mean"], yerr=plot_df["PSNR_std"], capsize=3, color="seagreen")
    axes[1].set_title("PSNR imagen (decode → SPECT)")
    axes[1].set_xticks(x)
    axes[1].set_xticklabels(labels, rotation=45, ha="right")

    axes[2].bar(x, plot_df["SSIM_mean"], yerr=plot_df["SSIM_std"], capsize=3, color="coral")
    axes[2].set_title("SSIM imagen")
    axes[2].set_xticks(x)
    axes[2].set_xticklabels(labels, rotation=45, ha="right")

    fig.suptitle(f"Diagnóstico sampling — {variant_label} (label={'sí' if USE_LABEL else 'no'})", y=1.02)
    plt.tight_layout()
    plt.show()

    ft = plot_df[plot_df["method"] == "fixed_t"].sort_values("t_or_steps")
    if len(ft):
        fig, ax = plt.subplots(figsize=(7, 4))
        ax.plot(ft["t_or_steps"], ft["latent_mse_mean"], "o-", label="MSE latente")
        ax2 = ax.twinx()
        ax2.plot(ft["t_or_steps"], ft["PSNR_mean"], "s--", color="green", label="PSNR imagen")
        ax.set(xlabel="t", ylabel="MSE latente", title=f"1-paso desde z_gt ruidoso — {variant_label}")
        ax2.set_ylabel("PSNR imagen", color="green")
        ax.grid(True, alpha=0.3)
        plt.tight_layout()
        plt.show()


In [ ]:
diag_out = repo_root() / "result" / "stage2_frozen_v1_v2_summary"
diag_out.mkdir(parents=True, exist_ok=True)
tag = f"frozen_v1_v2_label{int(USE_LABEL)}"
diag_df.to_csv(diag_out / f"sampling_diagnosis_{tag}_per_subject.csv", index=False)
summary.to_csv(diag_out / f"sampling_diagnosis_{tag}_summary.csv", index=False)
print(f"Exportado en {diag_out}")

for variant_label in summary["variant_label"].unique():
    sub = summary[summary["variant_label"] == variant_label]
    ddim_rows = sub[sub["method"] == "ddim"].sort_values("t_or_steps")
    ft_high = sub[(sub["method"] == "fixed_t") & (sub["t_or_steps"] >= 500)]
    print(f"\n--- {variant_label} ---")
    if len(ddim_rows):
        print(f"DDIM-100 MSE latente mean: {ddim_rows['latent_mse_mean'].iloc[0]:.4f}")
    if len(ft_high) and len(ddim_rows) and ft_high["latent_mse_mean"].mean() > ddim_rows["latent_mse_mean"].iloc[0] * 2:
        print("→ Denoising 1-paso falla en t altos: priorizar reentreno sobre más steps DDIM.")


**Valoración de superposición de mascara basal ganglia sobre un caso particular MRI y SPECT**

In [ ]:
# Sobreposición máscara basal ganglia sobre MRI y SPECT (sujeto 3102, cortes axiales z ∈ [63, 90])
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import nibabel as nib
import numpy as np

if "DATA_ROOT" not in globals():
    here = Path.cwd().resolve()
    for candidate in (here, here.parent):
        if (candidate / "dataset.py").is_file() and (candidate / "viz_pairs.py").is_file():
            _root = candidate
            break
    else:
        raise RuntimeError(
            "No se encontró la raíz del repo. Ejecuta la celda de configuración o abre el notebook desde IL-CLDM/."
        )
    _root_str = str(_root)
    if _root_str not in sys.path:
        sys.path.insert(0, _root_str)
    if "code" in sys.modules and not hasattr(sys.modules["code"], "__path__"):
        del sys.modules["code"]
    DATA_ROOT = _root / "data"

from dataset import nifti_to_numpy
from viz_pairs import display_range

SUBJECT_ID = "3102"
Z_MIN, Z_MAX = 63, 90
MASK_PATH = DATA_ROOT / "mask_basal_ganglia.nii.gz"
MRI_PATH = DATA_ROOT / "whole_MRI" / f"{SUBJECT_ID}.nii.gz"
SPECT_PATH = DATA_ROOT / "whole_Abeta" / f"{SUBJECT_ID}.nii.gz"

bg_mask = nib.load(str(MASK_PATH)).get_fdata(dtype=np.float32)
bg_mask = (bg_mask > 0.5).astype(np.float32)

mri = nifti_to_numpy(str(MRI_PATH), zscore=True)
spect = nifti_to_numpy(str(SPECT_PATH), zscore=True)

if not (mri.shape == bg_mask.shape == spect.shape):
    raise ValueError(
        f"Shapes distintas: MRI {mri.shape}, SPECT {spect.shape}, máscara {bg_mask.shape}"
    )

z_slices = np.arange(Z_MIN, Z_MAX + 1)
print(
    f"Volumen {mri.shape} | cortes axiales z={Z_MIN}…{Z_MAX} ({len(z_slices)} slices) | "
    f"vox máscara total={int(bg_mask.sum()):,}"
)


def _overlay_slice(
    ax,
    vol,
    mask_vol,
    z,
    *,
    cmap="gray",
    vmin=None,
    vmax=None,
    mask_rgba=(1.0, 0.2, 0.2, 0.45),
):
    sl = vol[:, :, z].T
    msl = mask_vol[:, :, z].T > 0.5
    ax.imshow(sl, cmap=cmap, origin="lower", vmin=vmin, vmax=vmax)
    overlay = np.zeros((*sl.shape, 4), dtype=np.float32)
    overlay[msl, :3] = mask_rgba[:3]
    overlay[msl, 3] = mask_rgba[3]
    ax.imshow(overlay, origin="lower")
    ax.set_title(f"z={z}", fontsize=8)
    ax.axis("off")


def plot_mask_overlay_grid(vol, modality_label, *, symmetric=True, spect_hot=False):
    if spect_hot:
        vmin, vmax = display_range(vol, symmetric=False, percentiles=(5.0, 99.5))
        cmap = "hot"
    else:
        vmin, vmax = display_range(vol, symmetric=symmetric)
        cmap = "gray"

    ncols = 7
    nrows = int(np.ceil(len(z_slices) / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(2.2 * ncols, 2.2 * nrows))
    axes = np.atleast_2d(axes)

    for idx, z in enumerate(z_slices):
        r, c = divmod(idx, ncols)
        _overlay_slice(axes[r, c], vol, bg_mask, z, cmap=cmap, vmin=vmin, vmax=vmax)

    for idx in range(len(z_slices), nrows * ncols):
        r, c = divmod(idx, ncols)
        axes[r, c].axis("off")

    fig.suptitle(
        f"Sujeto {SUBJECT_ID} — {modality_label} + máscara basal ganglia "
        f"(cortes axiales z∈[{Z_MIN}, {Z_MAX}])",
        y=1.02,
    )
    plt.tight_layout()
    plt.show()


plot_mask_overlay_grid(mri, "MRI", symmetric=True)
plot_mask_overlay_grid(spect, "SPECT", spect_hot=True)